In [1]:
!pip install genesis-world==0.3.10

Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple


In [ ]:
import genesis as gs
import numpy as np
import math

# ===================== 1. SETUP =====================
gs.init(backend=gs.cuda, logging_level="warning")

# Increased dt to 0.002 (2ms) for collision stability as per logs
sim_dt = 0.002 
video_fps = 30
steps_per_frame = int(1.0 / (sim_dt * video_fps))

scene = gs.Scene(
    sim_options=gs.options.SimOptions(dt=sim_dt, substeps=10),
    show_viewer=False,
    renderer=gs.renderers.Rasterizer(),
)

# ===================== 2. TRACK & OBSTACLES =====================
scene.add_entity(gs.morphs.Plane())

OVAL_RADIUS = 15.0
STRAIGHT_LEN = 40.0
TRACK_WIDTH = 8.0
NUM_CONES = 60 

def get_oval_pos(dist, offset=0.0):
    perimeter = 2 * STRAIGHT_LEN + 2 * math.pi * OVAL_RADIUS
    d = dist % perimeter
    if d < STRAIGHT_LEN: return (OVAL_RADIUS + offset, -STRAIGHT_LEN/2 + d)
    d -= STRAIGHT_LEN
    if d < math.pi * OVAL_RADIUS:
        angle = (d / (math.pi * OVAL_RADIUS)) * math.pi
        return ((OVAL_RADIUS + offset) * math.cos(angle), STRAIGHT_LEN/2 + (OVAL_RADIUS + offset) * math.sin(angle))
    d -= math.pi * OVAL_RADIUS
    if d < STRAIGHT_LEN: return (-OVAL_RADIUS - offset, STRAIGHT_LEN/2 - d)
    d -= STRAIGHT_LEN
    angle = math.pi + (d / (math.pi * OVAL_RADIUS)) * math.pi
    return ((OVAL_RADIUS + offset) * math.cos(angle), -STRAIGHT_LEN/2 + (OVAL_RADIUS + offset) * math.sin(angle))

# Place Track Cones (Yellow)
for i in range(NUM_CONES):
    dist = (i / NUM_CONES) * (2 * STRAIGHT_LEN + 2 * math.pi * OVAL_RADIUS)
    for side in [-TRACK_WIDTH/2, TRACK_WIDTH/2]:
        cx, cy = get_oval_pos(dist, offset=side)
        scene.add_entity(
            gs.morphs.Cylinder(radius=0.3, height=1.0, pos=(cx, cy, 0.5)),
            surface=gs.surfaces.Default(color=(1.0, 1.0, 0.0), emissive=(0.3, 0.3, 0.0)),
            # FIXED: Friction must be >= 0.01
            material=gs.materials.Rigid(friction=0.01) 
        )

# Place Obstacles (Red Boxes)
obs_positions = [(15.0, 5.0), (12.0, 30.0), (-5.0, 25.0)]
for x, y in obs_positions:
    scene.add_entity(
        gs.morphs.Box(size=(1.5, 1.5, 1.5), pos=(x, y, 0.75)),
        surface=gs.surfaces.Default(color=(1.0, 0.0, 0.0), emissive=(0.4, 0.0, 0.0)),
        material=gs.materials.Rigid(friction=1.0),
    )

# ===================== 3. CAR & SENSORS =====================
start_x, start_y = get_oval_pos(0.0)
car = scene.add_entity(
    gs.morphs.URDF(file="assets/smart_car.urdf", fixed=False, pos=(start_x, start_y-5.0, 0.8)),
    material=gs.materials.Rigid(friction=4.0),
)

cam_lidar = scene.add_camera(res=(320, 100), pos=(0, 0, 1.5), lookat=(0, 10, 1.5), fov=110, far=40)
cam_chase = scene.add_camera(res=(640, 480), pos=(0,0,0), lookat=(0,0,0), fov=60, far=300)
cam_top = scene.add_camera(res=(640, 480), pos=(0,0,0), lookat=(0,0,0), up=(0, 1, 0), fov=60, far=300)

scene.build()

# ===================== 4. CONTROLLER =====================
def get_control_command(car_pos, car_yaw, t_param):
    pc, mask = cam_lidar.render_pointcloud(world_frame=True)
    if hasattr(pc, "detach"): pc = pc.detach().cpu().numpy()
    if hasattr(mask, "detach"): mask = mask.detach().cpu().numpy()
    
    valid_pts = pc[mask]
    steer_override, brake_trigger = 0.0, False
    
    if valid_pts.shape[0] > 0:
        cy, sy = math.cos(-car_yaw), math.sin(-car_yaw)
        rel = valid_pts - car_pos
        lx = rel[:, 0] * cy - rel[:, 1] * sy 
        ly = rel[:, 0] * sy + rel[:, 1] * cy 
        
        danger_mask = (ly > 0.5) & (ly < 12.0) & (np.abs(lx) < 2.5)
        if np.any(danger_mask):
            brake_trigger = True
            obs_x = lx[danger_mask][np.argmin(ly[danger_mask])]
            steer_override = -2.0 if obs_x > -0.2 else 2.0

    tx, ty = get_oval_pos(t_param + 4.0)
    path_yaw = math.atan2(tx - car_pos[0], ty - car_pos[1])
    yaw_err = (path_yaw - car_yaw + math.pi) % (2 * math.pi) - math.pi
    
    target_speed = 4.0 if brake_trigger else 12.0
    steer_cmd = steer_override if brake_trigger else yaw_err * 2.5
    return target_speed, np.clip(steer_cmd, -3.0, 3.0)

# ===================== 5. LOOP =====================
cam_chase.start_recording(); cam_top.start_recording()
t_param = 0.0

for step in range(int(10.0 / sim_dt)):
    qpos = car.get_qpos().detach().cpu().numpy()
    pos, yaw = qpos[:3], math.atan2(2.0*(qpos[3]*qpos[6] + qpos[4]*qpos[5]), 1.0 - 2.0*(qpos[5]**2 + qpos[6]**2))
    
    cam_lidar.set_pose(pos=(pos[0], pos[1], pos[2]+0.6), lookat=(pos[0]+5*math.sin(yaw), pos[1]+5*math.cos(yaw), pos[2]+0.6))
    
    spd, steer = get_control_command(pos, yaw, t_param)
    t_param += spd * sim_dt
    car.set_dofs_velocity(np.array([spd*math.sin(yaw), spd*math.cos(yaw), 0, 0, 0, steer]))
    
    scene.step()
    if step % steps_per_frame == 0:
        cam_chase.set_pose(pos=(pos[0]-8*math.sin(yaw), pos[1]-8*math.cos(yaw), 4), lookat=(pos[0], pos[1], 1))
        cam_top.set_pose(pos=(pos[0], pos[1], 40), lookat=(pos[0], pos[1], 0), up=(0, 1, 0))
        cam_chase.render(); cam_top.render()

cam_chase.stop_recording(save_to_filename="correct_chase.mp4", fps=video_fps)
cam_top.stop_recording(save_to_filename="correct_top.mp4", fps=video_fps)

[I 01/01/26 05:50:47.162 13337] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout


[Genesis] [05:50:50] [WARNING] Using a simulation timestep smaller than 2ms is not recommended for 'use_gjk_collision=False' as it could lead to numerically unstable collision detection.
